<h1> Project 1: Data Modeling <h1>

<h3>Set Directory<h3>

In [28]:
import os
new_directory = "C:\\Users\\Owner\\Repositories\\MEE-460"
os.chdir(new_directory)

<h3> Neural Network Model Training <h3>

In [29]:
# -*- coding: utf-8 -*-
"""
Created on Tue Mar 12 02:53:43 2024

@author: Owner
"""
import pickle as pkl
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from pickle import dump, load
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
import matplotlib.pyplot as plt
import seaborn as sns
import random as rn
import numpy as np
import os
import pandas as pd
import warnings
import joblib
warnings.simplefilter(action='ignore', category=FutureWarning)

pd.options.mode.chained_assignment = None
scores = {}
r2Train = {}
Training_df = pd.read_csv("Complete_X_Y_dataset.csv")
results = pd.DataFrame()
for seed in range(20):
    os.environ['PYTHONHASHSEED'] = '0'
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    np.random.seed(seed)
    rn.seed(seed)

    Xarray = np.array(Training_df[['H-', '>C<', '-CH<', 'CH2<', '(CH2)cyclo<', 'CH3-', '-C6-', "Weiner's Path", "Weiner's Polarity",
                      'MDEC-11', 'MDEC-12', 'MDEC-13', 'MDEC-14', 'MDEC-22', 'MDEC-23', 'MDEC-24', 'MDEC-33', 'MDEC-34', 'MDEC-44']])
    Yarray = np.array(Training_df[['density']])

    X = np.array(Training_df[['H-', '>C<', 'MDEC-44']])
    Y = np.array(Training_df[['density']])

    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.20, random_state=seed)

    scalerX = StandardScaler()
    scalerY = StandardScaler()
    scalerX.fit(X)
    scalerY.fit(Y.reshape(-1, 1))

    X_train = scalerX.transform(X_train)
    Y_train = scalerY.transform(Y_train.reshape(-1, 1))

    X_test = scalerX.transform(X_test)
    Y_test = scalerY.transform(Y_test)

    dump(scalerX, open('NNscalerX.pkl', 'wb'))
    dump(scalerY, open('NNscalerY.pkl', 'wb'))

    # Define the neural network
    nn = MLPRegressor(hidden_layer_sizes=(100,2),  activation='relu', solver='adam', alpha=0.001, batch_size='auto',
                      learning_rate='constant', learning_rate_init=0.01, power_t=0.5, max_iter=1000, shuffle=True,
                      random_state=None, tol=0.0001, verbose=False, warm_start=False, momentum=0.9,
                      nesterovs_momentum=True, early_stopping=False, validation_fraction=0.1, beta_1=0.9, beta_2=0.999,
                      epsilon=1e-08)

    # Train the neural network
    
    nn.fit(X_train, Y_train.ravel())
    ScoreTrain = nn.score(X_train, Y_train) 
    #rSquaredtrain = print('nn Train Score of' + str( seed ) + ' is ' + str(ScoreTrain))
    ScoreTest = nn.score(X_test, Y_test)
    #rSquaredtest = print('nn Test Score of' + str( seed ) + ' is ' + str(ScoreTest))
    scores[seed] = (ScoreTrain, ScoreTest)
    modelNumber = seed
    predictionTrain = nn.predict(X_train)
    R2score = r2_score(Y_train, predictionTrain)
   # Get the model's parameters
    params = nn.get_params()
    
    # Add the score to the parameters dictionary
    params['Train Score'] = ScoreTrain
    params['Test Score'] = ScoreTest
    params['R2 Score'] = R2score
    params['Model Number'] = modelNumber
    
    # Append the results to the DataFrame
    results = results._append(params, ignore_index=True)
    results.to_csv('nn_results.csv', index=False)
    
    
  
    
    
    dump(nn, open('C:/Users/Owner/Repositories/MEE-460/NN/NN_model_'+str(seed)+'.pkl', 'wb')) #desktop

    #sorted_scores = sorted(scores.items(), key=lambda x: (x[1][0], x[1][1]), reverse=True)
    # Select the top 10 pairs
    #top_10_scores = sorted_scores[:10]

    # Print the top 10 pairs
    #for i in range(10):
    #    print(f"Seed {top_10_scores[i][0]}: Train Score = {top_10_scores[i][1][0]}, Test Score = {top_10_scores[i][1][1]}")

    SeededDictNN = {'model': modelNumber,
                    'X_train': X_train, 
                    'X_test': X_test, 
                    'Y_train': Y_train, 
                    'Y_test': Y_test,
                    'Train Score': ScoreTrain,
                    'Test Score': ScoreTest}
    dump(SeededDictNN, open('C:/Users/Owner/Repositories/MEE-460/NN/nn_model_XY_TrainTest' + str(seed)+'.pkl', 'wb'))

# Stack the models in file 
df = pd.read_csv('nn_results.csv')

# Sort by 'R2 Score' in descending order to get the best scores
df = df.sort_values(by='R2 Score', ascending=False)

# Select the top 10 models
top_10_models = df[:10]
WorstModel = df[19:]
print(WorstModel)


   activation  alpha batch_size  beta_1  beta_2  early_stopping       epsilon  \
14       relu  0.001       auto     0.9   0.999           False  1.000000e-08   

   hidden_layer_sizes learning_rate  learning_rate_init  ...  shuffle  solver  \
14           (100, 2)      constant                0.01  ...     True    adam   

       tol  validation_fraction  verbose  warm_start  Train Score  Test Score  \
14  0.0001                  0.1    False       False    -0.001647   -0.008444   

    R2 Score  Model Number  
14 -0.001647            14  

[1 rows x 27 columns]


<h3>Neural Network Model Stacking<h3>

In [41]:

# Load the top 10 models and create a list of tuples for stacking

new_directory = "C:\\Users\\Owner\\Repositories\\MEE-460\\NN"
os.chdir(new_directory)
models = []
for _, row in top_10_models.iterrows():
    model = load(open(f'NN_model_{int(row["Model Number"])}.pkl', 'rb'))
    models.append((f'model_{int(row["Model Number"])}', model))

worst_model_data = load(open('nn_model_XY_TrainTest12.pkl', 'rb'))
X_train_worst = worst_model_data['X_train']
Y_train_worst = worst_model_data['Y_train']
X_test_worst = worst_model_data['X_test']
Y_test_worst = worst_model_data['Y_test']

# Initialize the stacking regressor
stacking_regressor = StackingRegressor(estimators=models, final_estimator=LinearRegression())

# Fit the stacking regressor to your data
# Replace 'X_train' and 'Y_train' with your actual training data

stacked_nnFit = stacking_regressor.fit(X_train_worst, Y_train_worst.ravel())
stacked_nnScore = stacked_nnFit.score(X_test_worst, Y_test_worst.ravel())
print(f"The R^2 score of the stacked model on the worst model's test data is {stacked_nnScore}")

The R^2 score of the stacked model on the worst model's test data is 0.11187757527132669


<h3>Directory Change for Random Forest Modeling<h3>

In [31]:
import os
new_directory = "C:\\Users\\Owner\\Repositories\\MEE-460"
os.chdir(new_directory)

<h3>Random Forest Modeling<h3>

In [6]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) #eliminates future version warnings 
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
Training_df = pd.read_csv("Complete_X_Y_dataset.csv")
rfresults = pd.DataFrame()
rfscores = {}
scoresrf = {}
r2Train = {}
for seed in range(20):    #trains 20 models. The seed determines how initilization and randomization of each model is done
    import os
    os.environ['PYTHONHASHSEED'] = '0'   #To rn.seed, this code keeps randomization and splitting concistent for each model run
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    import numpy as np
    np.random.seed(seed)
    import random as rn
    rn.seed(seed)
   # import tensorflow
    import seaborn as sns #These are the packages for the random forest, we must write the code for the neural network. 
    import matplotlib.pyplot as plt
    from sklearn.model_selection import train_test_split
    from sklearn.model_selection import RandomizedSearchCV
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import r2_score
    from pickle import dump
    
    # ******************** Data acquisition ***************************************
    #
    #Training_df = pd.read_csv("Complete_X_Y_dataset.csv")
    #Xarray = np.array(Training_df[['H-', '>C<', '-CH<' , 'CH2<' , '(CH2)cyclo<', 'CH3-' , '-C6-', "Weiner's Path",	"Weiner's Polarity",	'MDEC-11',	'MDEC-12',	'MDEC-13',	'MDEC-14',	'MDEC-22',	'MDEC-23',	'MDEC-24',	'MDEC-33',	'MDEC-34',	'MDEC-44']])
    #Must choose what variable are using to compare to the target, Y, density
   
    #Yarray = np.array(Training_df[['density']]) # Density
    #print(Xarray)
    #print(Yarray)
    
    Xrf = np.array(Training_df[['H-', '>C<', 'MDEC-44']])
    Yrf = np.array(Training_df[['density']])
    
    X_trainrf, X_testrf, Y_trainrf, Y_testrf = train_test_split(Xrf, Yrf, test_size=0.20, random_state=seed) #test size can be changed but not to beyond .30. This symbolizes %  of dataset used for testing, and the rest is training. 
    
    #Merge X_train, X_test, y_train, Y_test and save per iteration 
    scalerXrf = StandardScaler() #This is data normalization for data that changes larger orders of magnitude.
    scalerYrf = StandardScaler() #Data standardization must be done as it will impact the model if it is not.
    scalerXrf.fit(Xrf) #This fits the scaling to the X-Data
    scalerYrf.fit(Yrf.reshape(-1, 1)) #This fits the scaling to the Y-Data. The reshape(-1,1) is for: 
        
        
        
    X_trainrf = scalerXrf.transform(X_trainrf) 
    Y_trainrf = scalerYrf.transform(Y_trainrf.reshape(-1, 1))
    X_testrf = scalerXrf.transform(X_testrf)
    Y_testrf = scalerYrf.transform(Y_testrf)

    # RFSeededDictRF = {'X_train': X_train, 
    #                 'X_test': X_test, 
    #                 'Y_train': Y_train, 
    #                 'Y_test': Y_test}
    # dump(RFSeededDictRF, open('C:/Users/Owner/Repositories/MEE-460/RF/rf_model_XY_TrainTest' + str(seed), 'wb'))
   
    
    dump(scalerXrf, open('RFscalerX.pkl', 'wb'))
    dump(scalerYrf, open('RFscalerY.pkl', 'wb'))
    # Random forest hyperparamter optimization parameters
    # Number of trees in random forest
    n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)] #How man trees will be generated
    # Number of features to consider at every split
    max_features = [1.0, 'sqrt', 'log2']
    # Maximum number of levels in tree
    max_depth = [int(x) for x in np.linspace(10, 110, num = 11)] #How deep the branches can go
    max_depth.append(None)
    # Minimum number of samples required to split a node
    min_samples_split = [int(x) for x in np.linspace(2, 50, num = 25)]
    #Minimum number of samples required at each leaf node
    min_samples_leaf = [int(x) for x in np.linspace(1, 50, num = 1)]
    # Method of selecting samples for training each tree
    bootstrap = [True, False]
    # Create the random grid
    random_grid = {'n_estimators': n_estimators,
                   'max_features': max_features,
                   'max_depth': max_depth,
                   'min_samples_split': min_samples_split,
                   'min_samples_leaf': min_samples_leaf,
                   'bootstrap': bootstrap}
    #print(random_grid) #The above is a dictionary
    # Use the random grid to search for best hyperparameters
    # First create the base model to tune
    rf = RandomForestRegressor() 
    # Random search of parameters, using 3 fold cross validation, 
    # search across 100 different combinations, and use all available cores
    rf_random = RandomizedSearchCV(estimator = rf, param_distributions = random_grid, n_iter = 100, cv = 10, verbose=2, random_state=seed, n_jobs = -1, return_train_score=True) #This is the optimizing function that will create a better model. n_iter can be increased to 1000+
    # Fit the random search model
    rf_random.fit(X_trainrf, Y_trainrf.reshape(-1,))
    # Determine Model prediction for train and test
    cv_results = rf_random.cv_results_

    ScoreTrainrf = rf_random.score(X_trainrf, Y_trainrf) 
    #rSquaredtrain = print('nn Train Score of' + str( seed ) + ' is ' + str(ScoreTrain))
    ScoreTestrf = rf_random.score(X_testrf, Y_testrf)
    #rSquaredtest = print('nn Test Score of' + str( seed ) + ' is ' + str(ScoreTest))
    scoresrf[seed] = (ScoreTrainrf, ScoreTestrf)
    modelNumberrf = seed
    predictionTrainrf = rf_random.predict(X_trainrf)
    R2scorerf = r2_score(Y_trainrf, predictionTrainrf)
   # Get the model's parameters
    params = rf_random.get_params()
    
    # Add the score to the parameters dictionary
    params['Train Score'] = ScoreTrainrf
    params['Test Score'] = ScoreTestrf
    params['R2 Score'] = R2scorerf
    params['Model Number'] = modelNumberrf
    
    # Append the results to the DataFrame
    rfresults = rfresults._append(params, ignore_index=True)
    rfresults.to_csv('rf_results.csv', index=False)

    dump(rf_random, open('C:/Users/Owner/Repositories/MEE-460/RF/rf_model_'+str(seed)+'.pkl', 'wb'))


    SeededDictRF = {'model': modelNumberrf, 
                    'X_train': X_trainrf, 
                    'X_test': X_testrf, 
                    'Y_train': Y_trainrf, 
                    'Y_test': Y_testrf,
                    'Train Score': ScoreTrainrf,
                    'Test Score': ScoreTestrf}
    dump(SeededDictRF, open('C:/Users/Owner/Repositories/MEE-460/RF/rf_model_XY_TrainTest' + str(seed)+'.pkl', 'wb'))

# Stack the models in file 
rfdf = pd.read_csv('rf_results.csv')

# Sort by 'R2 Score' in descending order to get the best scores
rfdf = rfdf.sort_values(by='R2 Score', ascending=False)

# Select the top 10 models
top_10_modelsrf = rfdf[:10]
WorstModelrf = rfdf[19:]


Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each

<h3>Show Best and Worst Random Forest Models <h3>

In [36]:
top_10_modelsrf = rfdf[:10]
print(top_10_modelsrf[['R2 Score' , 'Model Number']])
WorstModelrf = rfdf[19:]
print(WorstModelrf[['R2 Score' , 'Model Number']])

    R2 Score  Model Number
12  0.815106            12
13  0.806325            13
0   0.797621             0
9   0.796136             9
16  0.793117            16
11  0.792467            11
2   0.789999             2
7   0.787072             7
5   0.780288             5
8   0.778628             8
    R2 Score  Model Number
15  0.713519            15


<h3>Random Forest Model Stacking<h3>

In [42]:

new_directory = "C:\\Users\\Owner\\Repositories\\MEE-460\\RF"
os.chdir(new_directory)
modelsrf = []
for _, row in top_10_models.iterrows():
    modelrf = load(open(f'rf_model_{int(row["Model Number"])}.pkl', 'rb'))
    modelsrf.append((f'model_{int(row["Model Number"])}', modelrf))



worst_model_datarf = load(open('rf_model_XY_TrainTest15.pkl', 'rb'))
X_train_worstrf = worst_model_datarf['X_train']
Y_train_worstrf = worst_model_datarf['Y_train']
X_test_worstrf = worst_model_datarf['X_test']
Y_test_worstrf = worst_model_datarf['Y_test']

# Initialize the stacking regressor
stacking_regressor = StackingRegressor(estimators=modelsrf, final_estimator=LinearRegression())

# Fit the stacking regressor to your data
# Replace 'X_train' and 'Y_train' with your actual training data
rfStack = stacking_regressor.fit(X_train_worstrf, Y_train_worstrf.ravel())
preStack_r2_score_rf = rfStack.score(X_test_worstrf, Y_test_worstrf)
print(f"The R^2 score of the stacked model on the worst model's test data is {preStack_r2_score_rf}")
Stack_rfFit = stacking_regressor.fit(X_test_worstrf, Y_test_worstrf.ravel())



The R^2 score of the stacked model on the worst model's test data is 0.687028481922072
